In [4]:
from IPython.display import display
import time
import hmac
import hashlib
import urllib.request
from urllib.parse import quote
import json
import pandas as pd

In [8]:
API_KEY = "9cc2976dc30ff6190b7b0485536cc483"
SECRET  = "61f2cb7a045b539c7a530af70a5be50cbff571f69e6dd8fbaa3dcc1fb8835ac8"

METHOD = "GET"
PATH   = "prediction/playerSeason"
QUERY  = {"p_no": "14770"}

def normalize_query(params: dict) -> str:
    safe = "-_.!~*'()"
    return "&".join(
        f"{quote(str(k), safe=safe)}={quote(str(params[k]), safe=safe)}"
        for k in sorted(params.keys())
    )

def make_signature(secret: str, payload: str) -> str:
    return hmac.new(
        secret.encode("utf-8"),
        payload.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

normalized = normalize_query(QUERY)
timestamp = str(int(time.time()))

payload = f"{METHOD}|{PATH}|{normalized}|{timestamp}"
signature = make_signature(SECRET, payload)

url = f"https://api.statiz.co.kr/baseballApi/{PATH}?{normalized}"

headers = {
    "X-API-KEY": API_KEY,
    "X-TIMESTAMP": timestamp,
    "X-SIGNATURE": signature,
}

req = urllib.request.Request(url, method=METHOD, headers=headers)

with urllib.request.urlopen(req, timeout=30) as resp:
    body = resp.read().decode("utf-8")

# JSON 변환
json_data = json.loads(body)

# 확인
display(json_data)



{'basic': {'list': [{'p_no': 14770,
    'leagueType': 10100,
    't_code': '6002',
    'p_name': '최승용',
    'year': '2023',
    'p_nameEN': 'Seung-Yong Choi',
    'G': '34',
    'GS': '20',
    'GR': '14',
    'GF': '4',
    'CG': '0',
    'SHO': '0',
    'W': '3',
    'L': '6',
    'S': '1',
    'HD': '0',
    'IP': 111,
    'ER': '49',
    'R': '54',
    'TBF': '479',
    'H': '116',
    '2B': '11',
    '3B': '1',
    'HR': '9',
    'BB': '34',
    'HP': '4',
    'IB': '0',
    'SO': '82',
    'ROE': '7',
    'BK': '0',
    'WP': '3',
    'ERA': 3.9699999999999998,
    'FIP': 4.02856,
    'WHIP': 1.35,
    'WAR': 1.692680769260149,
    'QS': '4',
    'p_position': 1},
   {'p_no': 14770,
    'leagueType': 10100,
    't_code': '6002',
    'p_name': '최승용',
    'year': '2024',
    'p_nameEN': 'Seung-Yong Choi',
    'G': '12',
    'GS': '6',
    'GR': '6',
    'GF': '1',
    'CG': '0',
    'SHO': '0',
    'W': '2',
    'L': '0',
    'S': '0',
    'HD': '1',
    'IP': 27,
    'ER': '18',
 

In [16]:
# json 저장
with open("최승용_14770.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=4)


In [14]:
# json 불러오기
with open("최승용_14770.json", "r", encoding="utf-8") as f:
    json_data = json.load(f) # 데이터명 'json_data'

In [15]:
# DataFrame 변환
basic_df = pd.DataFrame(json_data["basic"]["list"])
deepen_df = pd.DataFrame(json_data["deepen"]["list"])

display(basic_df.head())
display(deepen_df.head())

print(basic_df.columns)

,p_no,leagueType,t_code,p_name,year,p_nameEN,G,GS,GR,GF,...,SO,ROE,BK,WP,ERA,FIP,WHIP,WAR,QS,p_position
0,14770,10100,6002,최승용,2023,Seung-Yong Choi,34,20,14,4,...,82,7,0,3,3.97,4.02856,1.35,1.692681,4,1
1,14770,10100,6002,최승용,2024,Seung-Yong Choi,12,6,6,1,...,21,2,0,0,6.00,5.76126,1.63,0.079338,1,1
2,14770,10100,6002,최승용,2025,Seung-Yong Choi,23,23,0,0,...,71,6,0,4,4.41,4.36649,1.35,1.696985,9,1


,p_no,t_code,p_name,year,K9,BB9,KBB,HR9,OBP,SLG,OPS
0,14770,6002,최승용,2023,6.649,2.757,2.4118,0.730,0.326,0.362,0.688
1,14770,6002,최승용,2024,7.000,2.333,3.0000,2.000,0.358,0.552,0.910
2,14770,6002,최승용,2025,5.493,2.785,1.9722,0.619,0.325,0.379,0.704


Index(['p_no', 'leagueType', 't_code', 'p_name', 'year', 'p_nameEN', 'G', 'GS',
       'GR', 'GF', 'CG', 'SHO', 'W', 'L', 'S', 'HD', 'IP', 'ER', 'R', 'TBF',
       'H', '2B', '3B', 'HR', 'BB', 'HP', 'IB', 'SO', 'ROE', 'BK', 'WP', 'ERA',
       'FIP', 'WHIP', 'WAR', 'QS', 'p_position'],
      dtype='object')


In [10]:
deepen_df["year"] = basic_df["year"].values

merged_df = basic_df.merge(
    deepen_df[["year", "K9", "BB9", "KBB", "HR9", "OBP", "SLG", "OPS"]],
    on="year",
    how="left"
)

display(merged_df.head())
print(merged_df.columns)

,p_no,leagueType,t_code,p_name,year,p_nameEN,G,GS,GR,GF,...,WAR,QS,p_position,K9,BB9,KBB,HR9,OBP,SLG,OPS
0,14770,10100,6002,최승용,2023,Seung-Yong Choi,34,20,14,4,...,1.692681,4,1,6.649,2.757,2.4118,0.730,0.326,0.362,0.688
1,14770,10100,6002,최승용,2024,Seung-Yong Choi,12,6,6,1,...,0.079338,1,1,7.000,2.333,3.0000,2.000,0.358,0.552,0.910
2,14770,10100,6002,최승용,2025,Seung-Yong Choi,23,23,0,0,...,1.696985,9,1,5.493,2.785,1.9722,0.619,0.325,0.379,0.704


Index(['p_no', 'leagueType', 't_code', 'p_name', 'year', 'p_nameEN', 'G', 'GS',
       'GR', 'GF', 'CG', 'SHO', 'W', 'L', 'S', 'HD', 'IP', 'ER', 'R', 'TBF',
       'H', '2B', '3B', 'HR', 'BB', 'HP', 'IB', 'SO', 'ROE', 'BK', 'WP', 'ERA',
       'FIP', 'WHIP', 'WAR', 'QS', 'p_position', 'K9', 'BB9', 'KBB', 'HR9',
       'OBP', 'SLG', 'OPS'],
      dtype='object')


In [11]:
str_cols = ["p_name","p_nameEN","t_code"]
int_cols = [
    "p_no", "leagueType", "year", "p_position",
    "G", "GS", "GR", "GF", "CG", "SHO", "W", "L", "S", "HD", # 게임수, 선발, 구원, 마무리, 완투, 완봉, 승무세홀
    "ER", "R", "TBF", # 자책, 실점, 상대타자수
    "H", "2B", "3B", "HR",
    "BB", "HP", "IB", "SO", # IB(고의사구)
    "ROE", "BK", "WP", # 실책출루, 보크, 폭투
    "QS"
]
float_cols = [
    "IP", "ERA", "FIP", "WHIP", "WAR", # IP(이닝), FIP(수비무관평자)
    "K9", "BB9", "KBB", "HR9",
    "OBP", "SLG", "OPS" # 피출루율, 장타율, 옵스
]

merged_df[str_cols] = merged_df[str_cols].astype("object")

for c in int_cols:
    if c in merged_df.columns:
        merged_df[c] = pd.to_numeric(merged_df[c], errors="coerce").astype("int64")

for c in float_cols:
    if c in merged_df.columns:
        merged_df[c] = pd.to_numeric(merged_df[c], errors="coerce")

In [12]:
display(merged_df)

merged_df.dtypes


,p_no,leagueType,t_code,p_name,year,p_nameEN,G,GS,GR,GF,...,WAR,QS,p_position,K9,BB9,KBB,HR9,OBP,SLG,OPS
0,14770,10100,6002,최승용,2023,Seung-Yong Choi,34,20,14,4,...,1.692681,4,1,6.649,2.757,2.4118,0.730,0.326,0.362,0.688
1,14770,10100,6002,최승용,2024,Seung-Yong Choi,12,6,6,1,...,0.079338,1,1,7.000,2.333,3.0000,2.000,0.358,0.552,0.910
2,14770,10100,6002,최승용,2025,Seung-Yong Choi,23,23,0,0,...,1.696985,9,1,5.493,2.785,1.9722,0.619,0.325,0.379,0.704


p_no            int64
leagueType      int64
t_code         object
p_name         object
year            int64
p_nameEN       object
G               int64
GS              int64
GR              int64
GF              int64
CG              int64
SHO             int64
W               int64
L               int64
S               int64
HD              int64
IP            float64
ER              int64
R               int64
TBF             int64
H               int64
2B              int64
3B              int64
HR              int64
BB              int64
HP              int64
IB              int64
SO              int64
ROE             int64
BK              int64
WP              int64
ERA           float64
FIP           float64
WHIP          float64
WAR           float64
QS              int64
p_position      int64
K9            float64
BB9           float64
KBB           float64
HR9           float64
OBP           float64
SLG           float64
OPS           float64
dtype: object